# Object & Class di Python

- Class adalah sebuah blueprint atau cetakan yang membungkus atribut dan method bahkan class lain.
- Object adalah hasil dari class tersebut, maksudnya pembuatan object = pemanggilan class. Object ini akan memiliki seluruh atribut dan method yang ada pada class tersebut

### 1. Instansiasi Object

```
1. __new__(cls, ...): Contructor sebenarnya. berfungsi untuk membuat dan mengembalikan instance objek baru di memori
2. __init__(self, ...): Initializer. bertugas mengisi atribut ke dalam instance yang telah dibuat oleh __new__

Tujuan __new__ agar setiap instansiasi objek dari class A akan tersimpan di 1 memori saja (tidak membuat memori baru)
```



In [5]:
class DatabaseConnection:
    _instance = None

    def __new__(cls, *args, **kwargs):
        if cls._instance is None:
            print("1. [__new__] Mengalokasikan memori dan membuat instance objek baru...")
            cls._instance = super().__new__(cls)
        return cls._instance

    def __init__(self, db_name):
        print(f"2. [__init__] Menginisialisasi atribut 'db_name': {db_name}")
        self.db_name = db_name

db1 = DatabaseConnection('production_db')
db2 = DatabaseConnection('analytics_db')

print(f"Apakah db1 dan db2 merujuk ke instance yang sama? {db1 is db2}")

1. [__new__] Mengalokasikan memori dan membuat instance objek baru...
2. [__init__] Menginisialisasi atribut 'db_name': production_db
2. [__init__] Menginisialisasi atribut 'db_name': analytics_db
Apakah db1 dan db2 merujuk ke instance yang sama? True


In [11]:
# best praktice
class DatabaseConnection:
    _instance = None
    _initialized = False

    def __new__(cls, *args, **kwargs):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance

    def __init__(self, db_name, host="localhost", port=5432):
        if not DatabaseConnection._initialized:
            self.db_name = db_name
            self.host = host
            self.port = port
            DatabaseConnection._initialized = True

db1 = DatabaseConnection("production_db", "192.168.1.1", 5432)
print(db1.__dict__)
db2 = DatabaseConnection("test_db", "localhost", 3306)
print(db2.__dict__)

{'db_name': 'production_db', 'host': '192.168.1.1', 'port': 5432}
{'db_name': 'production_db', 'host': '192.168.1.1', 'port': 5432}


### 2. Dynamic Attribut Lookup
```
Setiap instance objek secara default menyimpan atribut tingkat instance didalam sebuah internal bernama __dict__

In [3]:
class Developer:
    company = "AA Studio"
    
    def __init__(self, nama, stack):
        self.nama = nama
        self.stack = stack

dev = Developer("Fathir", ["Pyhton","React_js"])

# menampilkan dict internal instance (level instance)
print(dev.__dict__)

# menambah atribut baru
dev.experience_years = 2
print(dev.__dict__)

# mencari atribut nama dam company
print(dev.nama)
print(dev.company)

# menampilkan semua atribut di level class
print(Developer.__dict__)

# mencari atribut company
print(Developer.company)

{'nama': 'Fathir', 'stack': ['Pyhton', 'React_js']}
{'nama': 'Fathir', 'stack': ['Pyhton', 'React_js'], 'experience_years': 2}
Fathir
AA Studio
{'__module__': '__main__', 'company': 'AA Studio', '__init__': <function Developer.__init__ at 0x000001FC2B280550>, '__dict__': <attribute '__dict__' of 'Developer' objects>, '__weakref__': <attribute '__weakref__' of 'Developer' objects>, '__doc__': None}
AA Studio


__Catatan__
```
nama_instance.__dict__ (Atribut pada level instance)
nama_instance.nama_atribut (Mencari dan menampilkan atribut instance) # bisa mencari atribut di luar init tapi tidak dalam function lain

nama_class.__dict__ (Atribut pada level class)
nama_class.nama_atribut (Mencari dan menampilkan atribut class) # tidak bisa mencari atribut di dalam function dan init
```

# 3. Optimasi Memori
```
Penggunaan __dict__ mengonsumsi alokasi memori yang besar jika program mengintansiasi ribuan objek.
__slots__ digunakan untuk membatasi atribut yang diizinkan sekaligus mematikan pembentukan __dict__
```

In [12]:
import sys

class StandardUser:
    def __init__(self, user_id, email):
        self.user_id = user_id
        self.email = email

class SlottedUser:
    __slots__ = ['user_id', 'email'] 

    def __init__(self, user_id, email):
        self.user_id = user_id
        self.email = email

u1 = StandardUser(101, "user@test.com")
u2 = SlottedUser(101, "user@test.com")

# cara melihat dan mengambil atribut instance u2
print(u2.__slots__)

for attr in u2.__slots__:
    print(f"{attr}: {getattr(u2, attr)}")

print(f"Ukuran memori StandardUser : {sys.getsizeof(u1) + sys.getsizeof(u1.__dict__)} bytes")
print(f"Ukuran memori SlottedUser  : {sys.getsizeof(u2)} bytes")

['user_id', 'email']
user_id: 101
email: user@test.com
Ukuran memori StandardUser : 152 bytes
Ukuran memori SlottedUser  : 48 bytes


# 4. @classmethod, @staticmethod, dan Alternative Constructors
```
- @classmethod (cls): Menerima referensi kelas sebagai parameter pertama. Sering digunakan untuk membuat Alternative Constructor
- @staticmethod: Method yang tidak bisa menerima referensi instance (self) maupun class (cls). Berfungsi sebagai utilitas independen di dalam namespace kelas

Alternative construktor seperti kita membuat instance namun dari file lain, seperti json dll.

staticmethod memungkinkan kita untuk memanggil fungsi atau method tanpa membuat instance

In [12]:
import json

class ConfigLoader:
    def __init__(self, config_dict):
        self.config = config_dict

    @classmethod
    def from_json_file(cls, filepath):
        with open(filepath, 'r') as f:
            data = json.load(f)
        return cls(data)

    @classmethod
    def from_raw_string(cls, raw_str):
        data = dict(item.split('=') for item in raw_str.split(','))
        return cls(data)

    @staticmethod
    def validate_key(key):
        return isinstance(key, str) and key.isupper()

In [1]:
# contoh alternative construktor
import json

class User:
    def __init__(self, username, role):
        self.username = username
        self.role = role
        
    @classmethod
    def from_json(cls, json_string):
        data = json.loads(json_string)
        return cls(data['username'], data['role'])

    @classmethod
    def from_string(cls, text_string):
        username, role = text_string.split("-")
        return cls(username, role)

u1 = User("Fathir", "Developer")

json_data = '{"username": "Rhey", "role": "Admin"}'
u2 = User.from_json(json_data)

u3 = User.from_string("Ahmad-Guest")

print(f"u1: {u1.username} ({u1.role})")
print(f"u2: {u2.username} ({u2.role})")
print(f"u3: {u3.username} ({u3.role})")

u1: Fathir (Developer)
u2: Rhey (Admin)
u3: Ahmad (Guest)


In [4]:
# Memanggil staticmethod tanpa membuat instance
class User:
    def __init__(self, username, email):
        self.username = username
        self.email = email

    @staticmethod
    def validate_email(email):
        return "@" in email and "." in email


is_valid = User.validate_email("fathir@test.com")
print(f"Apakah email valid? {is_valid}")

Apakah email valid? True


### Soal 1.1: Toko Buku (Instance Atribut & Method)
Buatlah sebuah kelas bernama Buku dengan ketentuan:

- Memiliki constructor (__init__) untuk menerima parameter judul, penulis, dan harga.
- Buatlah sebuah instance method bernama tampilkan_info(self) yang mencetak informasi lengkap buku tersebut.
- Buatlah 2 instance objek dari kelas Buku dengan data bebas, lalu panggil method tampilkan_info() pada masing-masing objek.

In [14]:
class Buku:
    def __init__(self, judul, penulis, harga):
        self.judul = judul
        self.penulis = penulis
        self.harga = harga
    
    def tampilkan_info(self):
        print(f"Judul: {self.judul} | Penulis: {self.penulis} | Harga: {self.harga}")

buku1 = Buku("Aroma Karsa", "Dee Lestari", 200000)
buku2 = Buku("Fall", "Albert Camus", 98000)

buku1.tampilkan_info()
buku2.tampilkan_info()

Judul: Aroma Karsa | Penulis: Dee Lestari | Harga: 200000
Judul: Fall | Penulis: Albert Camus | Harga: 98000


### Soal 1.2: Sistem Keranjang Belanja (Modifikasi Atribut Langsung)
Buatlah kelas bernama KeranjangBelanja dengan ketentuan:
```
- __init__ menerima parameter pemilik dan memiliki atribut daftar_item berupa list kosong ([]).
- Buat method tambah_item(self, nama_barang) untuk memasukkan barang ke dalam daftar_item.
- Buat objek keranjang atas nama kamu sendiri, tambahkan 3 barang belanjaan secara bergantian, lalu cetak isi keranjang tersebut.
```

In [21]:
class KeranjangBelanja:
    def __init__(self, pemilik, daftar_item=None):
        self.pemilik = pemilik
        
        if daftar_item is None:
            self.daftar_item = []
        else:
            self.daftar_item = daftar_item

    def tambah_item(self, barang):
        self.daftar_item.append(barang)

keranjang = KeranjangBelanja('Fathir')
keranjang.tambah_item("Amerikano")
keranjang.tambah_item("Novel")
keranjang.tambah_item("Pencil")

print(keranjang.daftar_item)

['Amerikano', 'Novel', 'Pencil']


### Soal 2.1: Parser Data Karyawan (Alternative Constructor)
Buatlah kelas Karyawan dengan ketentuan:

- Constructor utama (__init__) menerima nama dan gaji (integer).
- Buat sebuah Alternative Constructor dengan @classmethod bernama dari_string(cls, teks_format) yang menerima string dengan format "Nama-Gaji" (contoh: "Fathir-7500000"), memecahnya, lalu mengembalikan instance kelas Karyawan.
- Uji pembuatan objek menggunakan constructor utama dan Alternative Constructor dari_string.

In [6]:
class Karyawan:
    def __init__(self, nama, gaji):
        self.nama = nama
        self.gaji = int(gaji)

    @classmethod
    def from_string(cls, teks_string):
        nama, gaji = teks_string.split("-")
        return cls(nama, gaji)

# Pengujian
teks = "Liona-6000000"
karyawan1 = Karyawan('Fathir', 8000000)
karyawan2 = Karyawan.from_string(teks)

print(f"{karyawan1.nama} | {karyawan1.gaji}")
print(f"{karyawan2.nama} | {karyawan2.gaji}")

Fathir | 8000000
Liona | 6000000


### Soal 2.2: Sistem Validasi Nilai Ujian (Static Method)
Buatlah kelas UjianMahasiswa dengan ketentuan:
- Memiliki constructor untuk menyimpan nama dan nilai.
- Buat sebuah @staticmethod bernama cek_kelulusan(nilai) yang mengembalikan True jika nilai $\ge$ 70 dan False jika di bawah 70.
- Method ini harus bisa dipanggil langsung dari kelasnya tanpa harus membuat instance objek terlebih dahulu (UjianMahasiswa.cek_kelulusan(65)).

In [11]:
class UjianMahasiswa:
    def __init__(self, nama, nilai):
        self.nama = nama
        self.nilai = nilai

    @staticmethod
    def cek_kelulusan(nilai):
        if nilai >= 70:
            return True
        else:
            return False

print(UjianMahasiswa.cek_kelulusan(80))

True


### Soal 3.1: Pengaturan Aplikasi (Singleton Pattern via __new__)
Buatlah kelas AppConfig dengan ketentuan:
```
- Menerapkan Singleton Pattern menggunakan __new__ agar kelas ini hanya bisa dibuat 1 instance saja di seluruh program.
- Gunakan flag pengunci (_initialized) di dalam __init__ agar parameter konfigurasi (misal: app_name dan version) tidak tertimpa saat kelas dipanggil untuk kedua kalinya.
- Uji dengan membuat dua variabel (config1 dan config2), lalu buktikan dengan operasi is (config1 is config2 harus bernilai True).
```

In [15]:
class AppConfig:
    _instance = None
    _initialized = False

    def __new__(cls, *args, **kwargs):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance

    def __init__(self, app_name, version):
        if AppConfig._initialized == False:
            self.app_name = app_name
            self.version = version
            AppConfig._initialized = True

config1 = AppConfig('readverse', '1.2')
config2 = AppConfig('bacabuku', '1.1')

print(config2.app_name) # Hasil tetap readverse karena config 2 menunjuk ke memori yang sama

readverse


### Soal 3.2: Optimasi Memori Sensor IoT (__slots__)
Bayangkan sebuah sistem IoT yang membaca jutaan data sensor secara real-time. Buatlah kelas DataSensor dengan ketentuan:

- Menerapkan __slots__ yang hanya mengizinkan 3 atribut: id_sensor, suhu, dan waktu.
- Buat constructor untuk mengisi ketiga atribut tersebut.
- Coba buktikan bahwa instance kelas ini tidak memiliki atribut __dict__ (jika dipanggil akan memunculkan AttributeError), lalu tuliskan cara alternatif untuk membaca atributnya menggunakan perulangan pada __slots__.

In [21]:
import sys
class DataSensor:
    __slots__ = ['id_sensor','suhu','waktu']
    def __init__(self, id_sensor, suhu, waktu):
        self.id_sensor = id_sensor
        self.suhu = suhu
        self.waktu = waktu

data1 = DataSensor('101', 24, 13.00)
data1.id_sensor

for attr in data1.__slots__:
    print(f"{attr} : {getattr(data1, attr)}")

id_sensor : 101
suhu : 24
waktu : 13.0
